# 📈 SEC Financial Data Pipeline & RAG Demo
### Đồ Án Kỹ Thuật Dữ Liệu & AI RAG Tài Chính
---
Notebook này minh họa toàn bộ luồng xử lý dữ liệu qua kiến trúc **Medallion Architecture (Bronze -> Silver -> Gold)** phục vụ AI cho 7 tập đoàn công nghệ lớn: **AAPL, AMZN, GOOGL, META, MSFT, NVDA, TSLA**.

## 1. Nạp và kiểm tra dữ liệu Parquet đã làm sạch (Tầng Silver)
Chuẩn hóa mã CIK 10 chữ số, loại bỏ trùng lặp và lưu trữ dưới định dạng nén Snappy Parquet.

In [ ]:
import pandas as pd
from pathlib import Path

parquet_path = Path("../data/02_staging/companies_clean.parquet")
df = pd.read_parquet(parquet_path)
print(f"Tổng số công ty đã chuẩn hóa: {len(df):,}")
df[df['is_major']].head(10)

## 2. Kiểm tra văn bản Markdown bóc tách cho AI (Item 1A Risk Factors)
Bóc tách thẻ HTML/XBRL rác, trích xuất chính xác phần rủi ro kinh doanh (Risk Factors) của cả 7 tập đoàn công nghệ lớn.

In [ ]:
staging_dir = Path("../data/02_staging/sec_filings")
md_files = sorted(staging_dir.glob("*/*_risk_factors.md"))

summary = []
for f in md_files:
    text = f.read_text(encoding="utf-8")
    summary.append({
        "Ticker": f.parent.name,
        "File": f.name,
        "Dung lượng (KB)": round(f.stat().st_size / 1024, 1),
        "Số ký tự": f"{len(text):,}",
        "Đoạn mở đầu": text.split("\n\n")[1][:100] if len(text.split("\n\n")) > 1 else text[:100]
    })

pd.DataFrame(summary)

## 3. Kiểm tra các Chunks chuẩn bị nạp vào Vector DB (Tầng Gold)
Phân mảnh văn bản thành semantic chunks (500 từ + overlap 100 từ) kèm metadata chuẩn hóa (`chunk_id`, `ticker`, `year`, `word_count`).

In [ ]:
chunks_df = pd.read_parquet("../data/03_primary/chunks/all_chunks.parquet")
print(f"Tổng số chunks toàn bộ 7 công ty: {len(chunks_df):,}")
print("\nPhân bố số lượng chunk theo từng công ty:")
print(chunks_df['ticker'].value_counts())

chunks_df[['chunk_id', 'ticker', 'year', 'chunk_index', 'total_chunks', 'word_count', 'text']].head(5)